In [56]:
# Agentic AI with LangChain Tutorial for UNLV AI & Data Science Club

In [57]:
# import necessary libraries
# pip install langchain langchain-core langchain-community langchain-google-genai ddgs requests

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
import requests

from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
WEATHER_API_KEY = userdata.get("WEATHER_API_KEY")

In [58]:
# declare tools

# weather tool
@tool
def get_weather(city: str) -> dict:
    """
    Get current weather for a given city using WeatherAPI.com.
    Returns structured weather data as a dictionary.
    """

    url = "https://api.weatherapi.com/v1/current.json"

    params = {
        "key": WEATHER_API_KEY,
        "q": city,
        "aqi": "no"
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        return {
            "error": True,
            "status_code": response.status_code,
            "message": response.text
        }

    data = response.json()

    return {
        "error": False,
        "city": data["location"]["name"],
        "region": data["location"]["region"],
        "country": data["location"]["country"],
        "temperature_c": data["current"]["temp_c"],
        "condition": data["current"]["condition"]["text"],
        "wind_kph": data["current"]["wind_kph"],
        "humidity": data["current"]["humidity"]
    }

In [59]:
# predefined search tool
search = DuckDuckGoSearchRun()

In [60]:
# aggregate all tools into one variable
tools = [get_weather, search]

In [61]:
# declare LLM
llm = ChatGoogleGenerativeAI(
  model='gemini-3.1-flash-lite-preview',
  temperature=0.7,
  google_api_key=GEMINI_API_KEY
)

# create agent and give it llm and tools
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant. Your job is to check the weather and help the user plan a day trip to their desired city using the search tool."
)

In [ ]:
# call agent
question = input("User: ")

response = agent.invoke({
    "messages":[{"role": "user", "content": question}]},
    config={"recursion_limit": 8}
)

print("AI:", response['messages'][-1].content[0]['text'])